# Notebook 02: Simulation Walkthrough

This notebook walks through a complete dialogue simulation episode step by step.

We demonstrate:
1. Selecting a scenario and respondent profile
2. Instantiating a dialogue agent (CandidateSelectorAgent)
3. Running the DialogueRunner
4. Inspecting the transcript turn-by-turn
5. Visualising respondent trust and stress trajectories

In [ ]:
import sys
sys.path.insert(0, "../src")

from norm_dialogue_framework.data.synthetic_generator import SyntheticScenarioGenerator
from norm_dialogue_framework.agents import CandidateSelectorAgent, BaselineAgent, RuleAugmentedAgent
from norm_dialogue_framework.simulation.dialogue_runner import DialogueRunner
from norm_dialogue_framework.evaluation.evaluator import Evaluator
from norm_dialogue_framework.visualization.plots import plot_trust_stress_trajectories
import plotly.graph_objects as go
import pandas as pd

## 1. Select a scenario

In [ ]:
gen = SyntheticScenarioGenerator(seed=42, respondent_profiles=["anxious"])
scenario = gen.generate(n=1)[0]

print("Scenario Type:", scenario.scenario_type)
print("Respondent Profile:", scenario.respondent_profile)
print("Sensitivity:", scenario.sensitivity_level)
print()
print("Context:")
print(" ", scenario.context_summary)
print()
print("Communication Goals:")
for g in scenario.communication_goals:
    print(" -", g)
print()
print("Recommended Norms:", scenario.recommended_norms)

## 2. Instantiate agent and runner

The `CandidateSelectorAgent` generates multiple candidate utterances and selects
the one with the highest composite reward (ethical alignment + utility).

In [ ]:
agent = CandidateSelectorAgent(n_candidates=5, seed=42)
evaluator = Evaluator()

runner = DialogueRunner(
    agent=agent,
    max_turns=8,
    min_turns=4,
    seed=42,
    evaluator=evaluator
)

print("Agent strategy:", agent.strategy_name)

## 3. Run the simulation

In [ ]:
episode = runner.run(scenario)

print(f"Episode ID: {episode.episode_id}")
print(f"Total agent turns: {episode.total_turns}")
print(f"Final trust level: {episode.final_trust_level:.3f}")
print(f"Final stress level: {episode.final_stress_level:.3f}")
print(f"Completed: {episode.completed}")

## 4. Inspect the transcript

In [ ]:
print("=" * 70)
for turn in episode.turns:
    speaker = str(turn.speaker).upper()
    print(f"[{speaker}] {turn.utterance}")
    if turn.turn_metrics:
        m = turn.turn_metrics
        print(f"  composite={m.composite_score:.2f} | "
              f"ethical={m.ethical_alignment_score:.2f} | "
              f"utility={m.utility_score:.2f} | "
              f"coercion={m.coercion_risk:.2f}")
    if turn.respondent_trust_level is not None:
        print(f"  [trust={turn.respondent_trust_level:.2f}, stress={turn.respondent_stress_level:.2f}]")
    print()

## 5. Visualise trust and stress trajectories

In [ ]:
trajectory_data = [
    {
        "speaker": str(t.speaker),
        "trust": t.respondent_trust_level,
        "stress": t.respondent_stress_level,
    }
    for t in episode.turns
    if t.respondent_trust_level is not None
]

fig = plot_trust_stress_trajectories(trajectory_data, use_plotly=True)
fig.show()

## 6. Per-turn metrics summary

In [ ]:
rows = []
for turn in episode.turns:
    if turn.turn_metrics:
        m = turn.turn_metrics
        rows.append({
            "turn_id": turn.turn_id,
            "composite_score": m.composite_score,
            "ethical_alignment": m.ethical_alignment_score,
            "utility": m.utility_score,
            "coercion_risk": m.coercion_risk,
            "empathy_score": m.empathy_score,
            "information_yield": m.information_yield,
        })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

## 7. Evaluate the episode

In [ ]:
summary = evaluator.evaluate_episode(episode)

print("=== EPISODE SUMMARY ===")
print(f"Composite score:         {summary.composite_score:.3f}")
print(f"Ethical alignment score: {summary.ethical_alignment_score:.3f}")
print(f"Utility score:           {summary.utility_score:.3f}")
print()
print(f"Mean coercion risk:      {summary.mean_coercion_risk:.3f}")
print(f"Mean empathy score:      {summary.mean_empathy_score:.3f}")
print(f"Mean neutrality:         {summary.mean_neutrality_score:.3f}")
print(f"Mean information yield:  {summary.mean_information_yield:.3f}")
print()
print(f"Final trust:  {summary.final_trust_level:.3f}")
print(f"Final stress: {summary.final_stress_level:.3f}")